# 01 - Real SC4001 EEG Observation

This notebook builds the observation-side evidence for the later SBI workflow. It does not run a simulator, perform prior predictive checks, or train an NPE.

**Frozen compatibility choice:** the legacy prose says Hamming, but the fitting-target implementation actually calls Welch with a Hann window. Hann is therefore the primary result here; Hamming is shown only as a sensitivity comparison.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next(
    candidate for candidate in (cwd, *cwd.parents)
    if (candidate / "S4_sbi" / "configs" / "observation_sc4001.yaml").is_file()
)
sys.path.insert(0, str(ROOT / "S4_sbi" / "src"))

from sleep_sbi import (
    build_observation_bundle,
    load_observation_config,
    metric_classification_rows,
    select_representative_epochs,
)

FIGURE_DIR = ROOT / "outputs" / "observation_notebook_validation"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})

In [ ]:
config = load_observation_config()
bundle = build_observation_bundle(config)
bundle.validate()

metadata_rows = [
    ("subject", bundle.subject_id),
    ("channel", bundle.channel),
    ("sampling rate", f"{bundle.fs_hz:g} Hz"),
    ("unit", bundle.provenance["analysis_unit"]),
    ("reference", bundle.provenance["reference"]),
    ("recording duration", f"{bundle.provenance['recording_duration_s'] / 3600:.3f} h"),
    ("epoch duration", f"{bundle.epoch_duration_s:g} s"),
    ("random seed", bundle.provenance["random_seed"]),
    ("FOOOF", bundle.provenance["fooof_version"]),
]
display(pd.DataFrame(metadata_rows, columns=["field", "value"]))
assert bundle.provenance["fooof_version"] == "1.1.1"

## Labels and native epoch coordinates

The table reports labels exactly as stored in the annotation EDF. R&K Stage 3 and Stage 4 both map to AASM N3. The short label `3` is also explicitly mapped to N3, avoiding the old `3 -> N2` bug. Annotation time beyond the complete PSG epochs is clipped; original 30-second PSG epoch indices are retained.

In [ ]:
annotation_table = pd.DataFrame(bundle.diagnostics["annotation_table"])
display(annotation_table)
assert set(annotation_table.loc[annotation_table.raw_label.isin(["Sleep stage 3", "Sleep stage 4"]), "aasm_label"]) == {"N3"}
print("Warnings:")
for warning in bundle.warnings:
    print(" -", warning)

In [ ]:
stage_to_y = {"Unknown": -1, "W": 0, "REM": 1, "N1": 2, "N2": 3, "N3": 4}
stages = np.asarray(bundle.diagnostics["mapped_stages"])
hours = np.arange(len(stages)) * bundle.epoch_duration_s / 3600
stage_y = np.array([stage_to_y[stage] for stage in stages])

fig, ax = plt.subplots(figsize=(13, 3.2))
ax.step(hours, stage_y, where="post", color="#263238", linewidth=0.8)
n3_mask = stages == "N3"
ax.fill_between(hours, 4.25, 3.75, where=n3_mask, step="post", color="#d1495b", alpha=0.65, label="N3")
ax.set(yticks=list(stage_to_y.values()), yticklabels=list(stage_to_y), xlabel="Recording time (h)", ylabel="Stage", title="SC4001 hypnogram with native N3 epochs")
ax.invert_yaxis()
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_hypnogram.png", dpi=160)
plt.show()

## Epoch QC and representative retained N3 EEG

QC is applied to each native epoch independently. No discontinuous N3 epochs are concatenated before filtering or event detection.

In [ ]:
qc = pd.DataFrame(bundle.diagnostics["qc_rows"])
qc_summary = pd.DataFrame([
    {"set": "all complete PSG epochs", "n_epochs": bundle.provenance["recording_complete_epochs"]},
    {"set": "N3", "n_epochs": len(bundle.n3_epoch_indices)},
    {"set": "retained N3", "n_epochs": len(bundle.retained_epoch_indices)},
    {"set": "rejected N3", "n_epochs": len(bundle.rejected_epoch_indices)},
])
display(qc_summary)
display(pd.Series(bundle.provenance["rejection_reason_counts"], name="count").rename_axis("rejection_reason").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for retained, color, label in [(True, "#287271", "retained"), (False, "#d1495b", "rejected")]:
    values = qc.loc[qc.retained == retained, "peak_to_peak_uv"]
    axes[0].hist(values, bins=24, alpha=0.65, color=color, label=f"{label} (n={len(values)})")
axes[0].axvline(config.max_peak_to_peak_uv, color="black", linestyle="--", label="QC threshold")
axes[0].set(xlabel="Peak-to-peak amplitude (uV)", ylabel="N3 epochs", title="Epoch-level QC distribution")
axes[0].legend()
axes[1].scatter(qc.epoch_index, qc.peak_to_peak_uv, c=np.where(qc.retained, "#287271", "#d1495b"), s=12)
axes[1].axhline(config.max_peak_to_peak_uv, color="black", linestyle="--")
axes[1].set(xlabel="Native 30-s epoch index", ylabel="Peak-to-peak amplitude (uV)", title="QC in recording coordinates")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_epoch_qc.png", dpi=160)
plt.show()

In [ ]:
representative_indices = select_representative_epochs(bundle, n_epochs=4)
row_for_epoch = {int(epoch): row for row, epoch in enumerate(bundle.retained_epoch_indices)}
time_s = np.arange(bundle.segments.shape[1]) / bundle.fs_hz
fig, axes = plt.subplots(len(representative_indices), 1, figsize=(13, 7), sharex=True, sharey=True)
for ax, epoch_index in zip(axes, representative_indices):
    segment = bundle.segments[row_for_epoch[int(epoch_index)]]
    ax.plot(time_s, segment, color="#334e68", linewidth=0.65)
    ax.set_ylabel("uV")
    ax.set_title(f"Retained N3 epoch {epoch_index}", loc="left", fontsize=9)
axes[-1].set_xlabel("Time within native epoch (s)")
fig.suptitle("Representative SC4001 Fpz-Cz N3 EEG epochs", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(FIGURE_DIR / "03_representative_n3_epochs.png", dpi=160)
plt.show()

## Welch PSD: Hann primary, Hamming sensitivity

Each retained 30-second epoch is passed independently to Welch with a 4-second window and 1-second overlap. The aggregate is the arithmetic mean of epoch PSDs in `uV^2/Hz`.

In [ ]:
freq = bundle.psd.frequencies_hz
band = (freq >= 0.2) & (freq <= 20)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.1))
for row in bundle.psd.epoch_psd_hann_uv2_hz[::max(1, len(bundle.segments)//35)]:
    axes[0].semilogy(freq[band], row[band], color="#8ba6a9", alpha=0.28, linewidth=0.6)
axes[0].semilogy(freq[band], bundle.psd.aggregate_hann_uv2_hz[band], color="#102a43", linewidth=2.2, label="Hann aggregate")
axes[0].axvspan(*config.so_psd_band_hz, color="#d1495b", alpha=0.12, label="SO band")
axes[0].set(xlabel="Frequency (Hz)", ylabel="PSD (uV^2/Hz)", title="Epoch-level and aggregate PSD")
axes[0].legend()
axes[1].plot(freq[band], bundle.psd.normalized_hann[band], color="#102a43", linewidth=2, label="Hann (primary)")
axes[1].plot(freq[band], bundle.psd.normalized_hamming[band], color="#ee6c4d", linewidth=1.6, linestyle="--", label="Hamming sensitivity")
axes[1].axvline(bundle.summaries["so_peak_frequency_hz"].value, color="#d1495b", linestyle=":", label="SO peak")
axes[1].set(xlabel="Frequency (Hz)", ylabel="Normalized PSD (1/Hz)", title="Scale-invariant PSD shape")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_psd_hann_hamming.png", dpi=160)
plt.show()

hann = bundle.diagnostics["spectral_hann"]
hamming = bundle.diagnostics["spectral_hamming"]
sensitivity = pd.DataFrame([
    {"metric": key, "Hann_primary": hann[key], "Hamming_sensitivity": hamming[key], "delta_Hamming_minus_Hann": hamming[key] - hann[key]}
    for key in hann
])
display(sensitivity)

## Slow-oscillation events, interval variability, and morphology

The 0.2-4 Hz filter, DOWN-to-UP detector, IBI calculation, and waveform extraction all operate inside each retained epoch. Intervals crossing an epoch boundary are never created.

In [ ]:
so = bundle.diagnostics["slow_oscillation"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(so["waveform_time_s"], so["waveform_mean_z"], color="#7b2cbf", linewidth=2)
axes[0].fill_between(so["waveform_time_s"], so["waveform_mean_z"] - so["waveform_sem_z"], so["waveform_mean_z"] + so["waveform_sem_z"], color="#7b2cbf", alpha=0.2)
axes[0].axvline(0, color="black", linestyle=":")
axes[0].set(xlabel="Time from SO trough (s)", ylabel="Amplitude (z)", title=f"SO waveform morphology (n={len(so['waveform_snippets_z'])})")
axes[1].hist(so["ibi_s"], bins=25, color="#2a9d8f", alpha=0.8)
axes[1].set(xlabel="Within-epoch SO event interval (s)", ylabel="Intervals", title=f"IBI distribution; CV={bundle.summaries['ibi_cv'].value:.3f}")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "05_so_waveform_and_ibi.png", dpi=160)
plt.show()

## Observable-channel spindle events

The held-out detector uses an 11-15 Hz fourth-order Butterworth filter, a 200 ms moving-RMS envelope, a per-epoch mean + 1.5 SD threshold, 0.1 s merging, and 0.5-3.0 s duration limits.

In [ ]:
spindle = bundle.diagnostics["spindle"]
event = max(spindle["events"], key=lambda item: item["peak_envelope_uv"])
row = row_for_epoch[event["epoch_index"]]
center = (event["start_sample"] + event["stop_sample"]) // 2
half = int(3 * bundle.fs_hz)
start, stop = max(0, center - half), min(bundle.segments.shape[1], center + half)
local_time = (np.arange(start, stop) - center) / bundle.fs_hz
fig, axes = plt.subplots(2, 1, figsize=(12, 5.2), sharex=True)
axes[0].plot(local_time, bundle.segments[row, start:stop], color="#334e68", linewidth=0.7, label="Fpz-Cz")
axes[0].plot(local_time, spindle["filtered_uv"][row, start:stop], color="#e76f51", linewidth=1, label="11-15 Hz")
axes[0].set(ylabel="Amplitude (uV)", title=f"Representative spindle event in native epoch {event['epoch_index']}")
axes[0].legend()
axes[1].plot(local_time, spindle["envelope_uv"][row, start:stop], color="#2a9d8f", label="RMS envelope")
axes[1].axhline(spindle["threshold_uv"][row], color="black", linestyle="--", label="per-epoch threshold")
event_lo = (event["start_sample"] - center) / bundle.fs_hz
event_hi = (event["stop_sample"] - center) / bundle.fs_hz
axes[1].axvspan(event_lo, event_hi, color="#e9c46a", alpha=0.35, label="detected event")
axes[1].set(xlabel="Time from event center (s)", ylabel="Envelope (uV)", title=f"Density={bundle.summaries['spindle_density_per_min'].value:.3f}/min; mean duration={bundle.summaries['spindle_mean_duration_s'].value:.3f}s")
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "06_spindle_event.png", dpi=160)
plt.show()

## Single-channel SO-spindle PAC

SO phase (0.5-1.5 Hz) and spindle amplitude (10-14 Hz) are filtered independently within every epoch, with 2 seconds trimmed from each edge. The new field is `pac_up_down_ratio`; `T11_lag_ms` is only a legacy misnomer and is not emitted.

In [ ]:
pac = bundle.diagnostics["pac"]
phase = pac["phase_bin_centers_rad"]
fig = plt.figure(figsize=(11, 4.2))
ax1 = fig.add_subplot(1, 2, 1)
ax1.bar(phase, pac["mean_amplitude_uv"], width=2*np.pi/len(phase), color="#457b9d", alpha=0.8)
ax1.axvline(pac["preferred_phase_rad"], color="#d1495b", linestyle="--", label="preferred phase")
ax1.set(xlabel="SO phase (rad)", ylabel="Mean spindle envelope (uV)", title=f"Tort MI={pac['mi']:.6f}; UP/DOWN={pac['pac_up_down_ratio']:.3f}")
ax1.legend()
ax2 = fig.add_subplot(1, 2, 2, projection="polar")
closed_phase = np.r_[phase, phase[0]]
closed_amp = np.r_[pac["mean_amplitude_uv"], pac["mean_amplitude_uv"][0]]
ax2.plot(closed_phase, closed_amp, color="#457b9d", linewidth=2)
ax2.plot([pac["preferred_phase_rad"], pac["preferred_phase_rad"]], [0, closed_amp.max()], color="#d1495b", linestyle="--")
ax2.set_title("Preferred SO phase")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "07_pac.png", dpi=160)
plt.show()

## Observation summary and planned metric roles

This is an observation-side schema only. A simulator-shared `SummaryContractV1` is deliberately deferred to the next stage.

In [ ]:
summary_table = pd.DataFrame(bundle.summary_rows())
display(summary_table[["category", "name", "value", "unit", "band_hz", "algorithm", "aggregation", "valid", "invalid_reason"]])
display(pd.DataFrame(metric_classification_rows()))

safe_payload = bundle.publication_safe_dict()
assert "segments" not in safe_payload and "psd" not in safe_payload
assert "T11_lag_ms" not in bundle.summaries
assert bundle.segments.shape[1] == int(bundle.fs_hz * bundle.epoch_duration_s)
assert len(bundle.retained_epoch_indices) + len(bundle.rejected_epoch_indices) == len(bundle.n3_epoch_indices)
print(f"Generated validation figures: {FIGURE_DIR}")